# DAIS 2026 — Deployment & Setup

Short checklist to get a workspace ready for the DAIS 2026 runbooks
(`Genie.ipynb`, `AgentBricks.ipynb`, `LakebaseApps.ipynb`, `MLflow.ipynb`).
All four run against a single `all`-target deployment.

## 1. Workspace prerequisites

The `all` target touches every product. Confirm with your account team
**before** deploying — missing entitlements fail mid-run and waste a slot.

- **Region & cloud**: AWS or Azure workspace, region with Lakebase Autoscaling
  GA (e.g. `us-west-2`, `us-east-1`, `eu-west-1`). Free Edition will *not* work
  for `all` — use the `free` target instead.
- **Unity Catalog**: metastore attached; current user can `CREATE CATALOG`
  (or the target catalog already exists and you have `ALL PRIVILEGES`).
- **Serverless compute**: enabled for jobs, notebooks, SQL warehouses, and
  Lakeflow pipelines (Admin Console → Compute → Serverless).
- **Model Serving / Foundation Model APIs**: `databricks-claude-sonnet-4-5`
  available pay-per-token (default `LLM_MODEL`; override with
  `--params "LLM_MODEL=…"` if your workspace only has Llama).
- **Agent Bricks** (account-level preview, request via field team):
  - Knowledge Assistants v2.1 API
  - Genie Spaces API
  - Multi-Agent Supervisor API
- **Databricks Apps**: enabled (Admin Console → Previews → Databricks Apps).
- **Lakebase Autoscaling**: enabled and quota for ≥1 project / 3 logical
  databases (Admin Console → Previews → Lakebase).
- **AI SQL functions**: `ai_parse_document`, `ai_classify`, `ai_extract`,
  `ai_summarize` available in DBSQL (default in supported regions).
- **ABAC** (governance beat): `CREATE POLICY` requires DBR 16.4+ on the SQL
  warehouse readers use; the `all`-target warehouses default to current
  channel so this is normally a no-op check. The deployer needs `MANAGE` on
  the demo catalog plus tag-policy admin to create governed tags.
- **Data Quality Monitoring** (governance beat): Admin Console → Previews →
  *"Data quality monitoring with anomaly detection (workspace level)"* must
  be ON, otherwise the `food_safety` monitor cell prints a warning and
  skips. `databricks-sdk >= 0.83` on the helper task's compute (default DBR
  ships ≥ 0.83 since mid-2025).
- **Databricks One / Genie app**: presenter user has the **Databricks One**
  entitlement so `/one` loads with the Genie surface. iOS/Android Genie app
  installed and signed into the same workspace for the mobile beat.

## 2. Local prerequisites

- `databricks` CLI ≥ 0.275 (`databricks -v`)
- `jq` on `PATH` (used by the `cleanup` script)
- Authenticated to the target workspace:

  ```bash
  databricks auth login --host https://<workspace>.cloud.databricks.com
  ```

## 3. Deploy

```bash
databricks bundle deploy -t all
databricks bundle run    caspers -t all --params "CATALOG=<your_catalog>"
```

The job runs ~30–45 min end-to-end. Watch the run in the Jobs UI; every
task creates resources that the runbooks reference. `Evaluation` is the
final task — when it goes green, you're demo-ready.

> **Cache bug:** if files don't appear synced after a redeploy, `rm -rf
> .databricks .bundle` and redeploy. See `AGENTS.md` for the full workflow.

## 4. Discover Domains (Beta, one-time UI setup)

Discover Domains is Beta and has **no public REST/SDK/Terraform API yet**, so
this step is manual. Skip if your workspace doesn't have the Discover preview
flag enabled — the rest of the demo still works.

In **Catalog → Discover → Domains → New domain**, create three domains and
pin the assets below (all live under `${CATALOG}` once the `all` job has run):

| Domain | Icon (pick the closest match) | Pin these assets |
|---|---|---|
| **Operations** | gear / clipboard / factory | `${CATALOG}.lakeflow` (orders, deliveries, locations) · `${CATALOG}.food_safety` (inspections) · Dashboards: *Operations*, *Delivery Performance & SLA* · Genie space: *Operations Intelligence* · App: *Casper's Ops Dashboard* · KA: *Inspection* |
| **Revenue & Customers** | dollar / trending-up / users | `${CATALOG}.lakeflow` (revenue, brands, customers) · Dashboard: *AI Agent Performance* · Genie space: *Revenue & Orders Intelligence* · KA: *Consultancy* |
| **Compliance & Safety** | shield / gavel / scale | `${CATALOG}.food_safety` (ai_parsed / classified / extracted / summarized inspections) · Genie space: *Menu & Safety Intelligence* · KAs: *Legal*, *Regulatory*, *Audit*, *Inspection* |

This survives `bundle destroy` and is independent of `_internal_state` —
delete domains manually from the Discover UI if you change catalogs.

### App thumbnails

The `Databricks_App_Refund_Manager` and `Operational_App` stages now upload
branded Casper's-red PNG thumbnails (🤖 Ops Dashboard, 💳 Refund Manager) via
`w.apps.update_app_thumbnail`, so the apps render with proper artwork inside
each domain. Generated at deploy time by `utils/app_thumbnails.py` (Pillow);
the call is best-effort — falls back silently on older SDKs or missing
fonts so it never blocks the job.

## 5. ABAC governance + data quality (governance demo beat)

`Environment_Helpers` writes four `CREATE POLICY` statements and one
schema-level `data_quality.create_monitor` call after Lakeflow finishes
materialising silver/gold. All four policies live in `${CATALOG}._security`
and are pure tag-driven (no hard-coded table lists), so they survive future
schema additions:

| # | Policy | Securable | UDF | Bypass group |
|---|---|---|---|---|
| 1 | `caspers_mask_pii` (column mask) | `lakeflow` schema | `mask_pii` | `caspers_pii_readers` |
| 2 | `caspers_region_filter` (row filter) | `simulator` schema | `filter_by_region` | `caspers_geo_admins` (+ `caspers_us_users` / `caspers_emea_users` for the matching half) |
| 3 | `caspers_high_value_gate` (row filter) | `lakeflow` schema | `filter_high_value` | `caspers_finance`, `caspers_managers` |
| 4 | `caspers_regulated_docs` (row filter) | `food_safety` schema | `filter_regulated_docs` | `caspers_compliance` |

**Demo groups** (optional but recommended). The policies reference seven
account-level groups via `is_account_group_member`. They are *not* required
for the policies to be created — `is_account_group_member` returns false
for non-existent groups, so the masking/filtering just applies to
*everyone* (workspace admins still bypass). Create them in **Account
console → User management → Groups** if you want to demonstrate the
unmasked perspective from a non-admin demo user:

```
caspers_pii_readers   caspers_finance      caspers_compliance
caspers_us_users      caspers_managers
caspers_emea_users    caspers_geo_admins
```

**Data quality monitor.** Schema-level anomaly detection on
`${CATALOG}.food_safety` surfaces completeness / freshness regressions in
Catalog Explorer → Data Quality. Skipped silently if the workspace preview
flag is off — flip the toggle in *Admin Console → Previews* and re-run the
`Environment_Helpers` task.

## 6. Pre-show warm-up (5 min before going on stage)

1. Open each runbook in the workspace and run its **pre-flight** cell with
   your `CATALOG` widget set — this prints fresh URLs and confirms every
   resource exists.
2. Click each app URL once (Ops Dashboard, Refund Manager) to dodge
   cold-start.
3. Click one sample question in each Genie space to warm
   `<catalog>-ops-warehouse` and `<catalog>-genie-warehouse`.
4. Open one Knowledge Assistant endpoint and the Supervisor endpoint to
   warm Model Serving.

## 7. Cleanup

```bash
databricks bundle run     cleanup -t all --var catalog=<your_catalog>
databricks bundle destroy -t all
```

`cleanup` is a **script**, not a job task — use `--var catalog=…`,
**not** `--params`. `destroy` only removes bundle-owned resources;
`cleanup` removes everything tracked in `<catalog>._internal_state.resources`
(Lakebase projects, apps, model endpoints, KAs, Genie spaces, …).